# 10 — Differential Privacy (Laplace & Gaussian Mechanisms)

Apply (ε,δ)-DP noise to synthetic data to prevent re-identification while preserving statistical utility.

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.inference.tier3_research import DifferentialPrivacy
import importlib, pandas as pd, numpy as np
from sqllocks_spindle.cli import _get_domain_registry

spindle = Spindle()
reg = _get_domain_registry()
mod, cls, _ = reg['retail']
domain = getattr(importlib.import_module(mod), cls)(schema_mode='3nf')
result = spindle.generate(domain=domain, scale='small', seed=42)
table = next(iter(result.tables))
df = result.tables[table]
print(f'Table: {table}  rows={len(df)}')

In [ ]:
# Apply Laplace mechanism with ε=1.0
dp = DifferentialPrivacy(epsilon=1.0, mechanism='laplace')
noised_df, dp_result = dp.apply(df)

print(f'Mechanism: {dp_result.mechanism}')
print(f'Columns noised: {dp_result.columns_noised}')
print(f'\nSensitivity per column:')
for col, sens in dp_result.actual_sensitivity.items():
    print(f'  {col}: L1 sensitivity = {sens:.2f}')

In [ ]:
# Compare statistics before and after DP
numeric = [c for c in df.select_dtypes(include='number').columns][:3]
for col in numeric:
    orig_mean = df[col].mean()
    dp_mean = noised_df[col].mean()
    print(f'{col}: orig_mean={orig_mean:.2f}  dp_mean={dp_mean:.2f}  delta={abs(orig_mean-dp_mean):.2f}')

In [ ]:
# Compare Laplace vs Gaussian at different ε values
print('\nPrivacy-Utility tradeoff (column: first numeric col):')
col = numeric[0] if numeric else None
if col:
    print(f'{"ε":<8} {"Mechanism":<12} {"Mean delta":>12} {"Std delta":>12}')
    for eps in [0.1, 0.5, 1.0, 2.0, 5.0]:
        for mech in ['laplace', 'gaussian']:
            dp2 = DifferentialPrivacy(epsilon=eps, mechanism=mech)
            nd, _ = dp2.apply(df)
            delta_mean = abs(df[col].mean() - nd[col].mean())
            delta_std = abs(df[col].std() - nd[col].std())
            print(f'{eps:<8.1f} {mech:<12} {delta_mean:>12.3f} {delta_std:>12.3f}')